# Optimal extraction: box vs. polynomial vs. Gaussian-process profiles

This tutorial extracts a simulated spectrum three ways and compares the results with the
true spectrum:

1. a fractional-pixel **box** aperture (`getFastSimpleSpectrum`),
2. **optimal extraction** (Marsh 1989) with the default **polynomial** spatial profile,
3. optimal extraction with an experimental **Gaussian-process** (GP) spatial profile.

Everything runs on synthetic data in a few seconds.

In [ ]:
import transitspectroscopy as ts
from transitspectroscopy import spectroscopy
print(ts.__version__)

## Simulate a spectrum

The detector image has a curved trace whose width changes along the dispersion direction,
an absorption line, Poisson and read noise, and two cosmic rays. The simulation integrates a
Gaussian over each pixel, which is *not* the model used by either extraction method.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ndtr

def simulate_spectrum(nrows=32, ncolumns=200, flux=300., read_noise=5., width=1.4,
                      seed=1, cosmic_rays=True):
    """Pixel-integrated Gaussian profile along a curved trace, with Poisson and read noise."""
    rng = np.random.default_rng(seed)
    t = np.arange(ncolumns) / (ncolumns - 1)
    centers = nrows / 2 + 2. * (t - .5) + .4 * np.sin(2 * np.pi * t)
    widths = width + .3 * np.sin(3 * np.pi * t)            # the PSF width changes with wavelength
    edges = np.arange(nrows + 1)[:, None] - .5
    profile = np.diff(ndtr((edges - centers) / widths), axis=0)
    spectrum = flux * (1 + .3 * t - .3 * np.exp(-((t - .4) / .02)**2))   # one absorption line
    expected = profile * spectrum
    data = rng.poisson(expected) + rng.normal(0., read_noise, expected.shape)
    if cosmic_rays:
        for column, amplitude in ((60, 1000.), (140, 500.)):
            data[int(round(centers[column])) + 2, column] += amplitude
    variance = expected + read_noise**2
    return data, variance, centers, spectrum

data, variance, centers, truth = simulate_spectrum()
plt.figure(figsize=(10, 2.5))
plt.imshow(data, aspect='auto', origin='lower', vmin=0, vmax=150, cmap='magma')
plt.plot(centers, color='white', lw=.8)
plt.xlabel('Column (dispersion)'); plt.ylabel('Row');

## Extract

`centroids` gives the trace position at every column. All three methods use a 7-pixel
aperture radius. The optimal extractions use the known pixel variances (`data_variance`)
and reject outliers (cosmic rays) at `nsigma=5`.

The polynomial profile needs enough terms to follow how the PSF changes along the
dispersion direction. Here the width oscillates 1.5 times across the detector, so we use
`polynomial_order=6` (six terms, maximum degree five); we come back to this below.

In [ ]:
radius = 7
box = spectroscopy.getFastSimpleSpectrum(data, centers, radius, backend='python')

poly, P_poly = spectroscopy.getOptimalSpectrum(
    data, centers, radius, ron=5., gain=1., nsigma=5., polynomial_spacing=1.,
    polynomial_order=6, data_variance=variance, backend='python', return_P=True)

gp, P_gp = spectroscopy.getOptimalSpectrum(
    data, centers, radius, ron=5., gain=1., nsigma=5., polynomial_spacing=1.,
    polynomial_order=6, data_variance=variance, backend='python', return_P=True,
    profile_method='gp', gp_options={'kernel': 'matern32', 'length_scale': 30., 'n_inducing': 16})

# Optimal spectra have three rows: column, flux and inverse variance.
column, poly_flux, poly_ivar = poly
gp_flux = gp[1]

## Compare with the truth

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(truth, 'k-', label='truth')
axes[0].plot(box, '.', ms=3, label='box')
axes[0].plot(poly_flux, '.', ms=3, label='optimal, polynomial')
axes[0].plot(gp_flux, '.', ms=3, label='optimal, GP')
axes[0].set_ylabel('Flux'); axes[0].legend(ncol=4, fontsize=9)
for flux, label in ((box, 'box'), (poly_flux, 'polynomial'), (gp_flux, 'GP')):
    axes[1].plot(column, (flux - truth) / truth * 100, '.', ms=3, label=label)
axes[1].set_ylabel('Residual (%)'); axes[1].set_xlabel('Column')
axes[1].set_ylim(-40, 40);

The box aperture is badly biased at the two columns hit by cosmic rays, because it sums
every pixel. Optimal extraction rejects them. Away from those columns all three methods agree
with the truth within the noise, and optimal extraction has the smallest scatter because it
down-weights pixels dominated by read noise. The gain grows with read noise and aperture size.

In [ ]:
good = np.ones(len(truth), bool); good[[60, 140]] = False
for flux, label in ((box, 'box'), (poly_flux, 'polynomial'), (gp_flux, 'GP')):
    scatter = np.std((flux[good] - truth[good]) / truth[good]) * 1e6
    print(f'{label:>10s}: scatter {scatter:6.0f} ppm; '
          f'worst cosmic-ray column error {np.max(np.abs(flux - truth)[~good] / truth[~good]) * 100:5.1f}%')
# Errors are conditional on the fitted profile: compare predicted and actual scatter.
pulls = (poly_flux - truth) * np.sqrt(poly_ivar)
print('std of polynomial (flux - truth)/error:', np.std(pulls[good]).round(2))

## When the profile model is too simple

Optimal extraction is only unbiased if the spatial profile is right. The error from a wrong
profile is a fraction of the flux, so it matters most for bright targets. For a version of
the same spectrum 20 times brighter, three polynomial terms cannot follow the changing PSF
width, and the spectrum is biased by many times its error bars:

In [ ]:
bright, bright_variance, _, bright_truth = simulate_spectrum(flux=6000.)
for terms in (3, 6):
    result = spectroscopy.getOptimalSpectrum(
        bright, centers, radius, ron=5., gain=1., nsigma=5., polynomial_spacing=1.,
        polynomial_order=terms, data_variance=bright_variance, backend='python')
    pull = (result[1] - bright_truth) * np.sqrt(result[2])
    print(f'{terms} terms: std of (flux - truth)/error = {np.std(pull[good]):.2f}')

Check the profile fit (for example with the residuals `data - flux * P`) before trusting
an optimal spectrum, or use the GP profile, whose flexibility is set by its length scale
rather than by a polynomial degree.

## The fitted spatial profiles

`return_P=True` returns the normalized spatial profile `P` (rows × columns). Fitting `P`
once (`spectroscopy.getP`) and passing it to every integration of a time series
(`getOptimalSpectrum(..., P=P)`) is much faster than refitting it each time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, P, title in zip(axes, (P_poly, P_gp), ('polynomial', 'GP')):
    ax.imshow(P, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(title); ax.set_xlabel('Column')
axes[0].set_ylabel('Row');

See {doc}`../guides/optimal-extraction` for the options, and
{doc}`shared-profile` for a profile trained once and reused through a moving trace.